# Lesson 3 — Tool Agent (CampusInfoToolAgent) with LangGraph + MCP → A2A Server → A2A Client

**Audience:** University faculty (Malaysia)  
**Goal:** Build a **tool-using agent** that:
1) **Chooses an MCP tool** based on a user question  
2) **Calls the tool via MCP (stdio server)**  
3) **Answers only using tool results**  
Then wrap it as an **A2A server** and test via an **A2A client**.

At the end, we do a **mini-orchestrator**: one question → call **two A2A agents** (CoursePolicyAgent + CampusInfoToolAgent) → merge final output.

---

## Prerequisites
- Lesson 1: `a2a_course_policy_agent.py` (server running on `127.0.0.1:9999`)
- Lesson 2: `campus_tools_mcp.py` + `campus_data.json` in the same folder

> **Important notebook tip:** we use top-level `await` (not `asyncio.run`) to avoid event-loop errors in Jupyter.


## 1. Setup

### 1.1 Install dependencies (run once)

We will use:
- `openai` for the LLM
- `langgraph` to define the agent flow (decide tool → call tool → draft answer)
- `mcp` for the MCP client/server protocol
- `a2a-sdk[http-server]` for A2A server


In [1]:
# If needed, uncomment and run:
!pip -q install openai "langgraph>=0.2.0" mcp "a2a-sdk[http-server]" httpx python-dotenv

In [2]:
import os
from dotenv import load_dotenv

_ = load_dotenv()

assert os.environ.get("OPENAI_API_KEY"), "Please set OPENAI_API_KEY in your environment."
print("✅ OPENAI_API_KEY found")

✅ OPENAI_API_KEY found


## 2. Ensure the MCP tool server file exists (from Lesson 2)

We will connect to the MCP server via **stdio**, meaning:
- The client spawns: `python campus_tools_mcp.py`
- Communication happens over stdin/stdout (no HTTP)

First, confirm the files are present.


In [3]:
from pathlib import Path

assert Path("campus_tools_mcp.py").exists(), "Missing campus_tools_mcp.py (create it in Lesson 2)."
assert Path("campus_data.json").exists(), "Missing campus_data.json (create it in Lesson 2)."

print("✅ Found campus_tools_mcp.py and campus_data.json")

✅ Found campus_tools_mcp.py and campus_data.json


## 3. Connect to MCP server (stdio) and list tools

We will:
- start the server as a subprocess
- list available tools and their input schemas


In [5]:
import sys
from mcp.client.stdio import StdioServerParameters

SERVER_PARAMS = StdioServerParameters(
    command=sys.executable,
    args=["-u", "A2A/campus_tools_mcp.py"],
)
from mcp.shared.memory import create_connected_server_and_client_session
import campus_tools_mcp  # imports mcp = FastMCP(...)

async def get_inmemory_session():
    # This runs the MCP server in-process (no subprocess)
    cm = create_connected_server_and_client_session(campus_tools_mcp.mcp, raise_exceptions=True)
    session = await cm.__aenter__()
    return session, cm  # remember to close: await cm.__aexit__(...)

session, session_cm = await get_inmemory_session()

tools_obj = await session.list_tools()
tools = [{"name": t.name, "description": t.description, "inputSchema": t.inputSchema} for t in tools_obj.tools]
print("✅ Tools:", [t["name"] for t in tools])

Processing request of type ListToolsRequest


✅ Tools: ['find_staff', 'get_office_hours', 'find_timetable', 'find_room', 'list_contacts']


Processing request of type CallToolRequest
Processing request of type CallToolRequest
Processing request of type CallToolRequest
Processing request of type CallToolRequest
Processing request of type CallToolRequest


## 4. Tool Agent contract (rules for reliability)

The agent must:
- choose **exactly one** tool
- provide arguments as JSON
- answer **only** from tool output

We standardize final output as:

**Answer**  
**Evidence** (tool + key fields)  
**Actions / Next steps**


## 5. Implement the Tool Agent with LangGraph

We build a 3-node graph:

1) `decide_tool` (LLM selects tool + args)  
2) `call_tool` (execute MCP tool)  
3) `draft_answer` (LLM formats final response using only tool output)


In [6]:
from openai import OpenAI
from langgraph.graph import StateGraph, END

import re
import json
from typing import Any, Dict, List

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
LLM_MODEL = "gpt-4o-mini"


def _build_schema_map(tools_meta: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    return {t["name"]: (t.get("inputSchema") or {}) for t in tools_meta}


def _extract_week(text: str) -> int | None:
    m = re.search(r"\bweek\s*(\d+)\b", text, flags=re.I)
    return int(m.group(1)) if m else None


def _extract_course_code(text: str) -> str | None:
    # e.g., SENG3200, DSAI2101
    m = re.search(r"\b[A-Z]{3,6}\d{3,5}\b", text)
    return m.group(0) if m else None


def _extract_staff_name(text: str) -> str | None:
    # simple: "for Dr Amina Rahman" or "for Prof Lim Wei Jian"
    m = re.search(r"\bfor\s+((Dr|Prof|Ms|Mr)\.?\s+[A-Za-z]+(?:\s+[A-Za-z]+){0,3})\b", text)
    return m.group(1).strip() if m else None


def _extract_room_token(text: str) -> str | None:
    # tokens like CS-Lab-2, LT-1, LIB-Meeting-1
    m = re.search(r"\b[A-Za-z]{2,5}-[A-Za-z]+(?:-[0-9]+)?\b|\bLT-\d+\b|\bCS-Lab-\d+\b|\bLIB-Meeting-\d+\b", text)
    return m.group(0) if m else None


def decide_tool(state: Dict[str, Any]) -> Dict[str, Any]:
    question = state["question"]
    tools_meta = state["tools_meta"]
    schema_map = _build_schema_map(tools_meta)

    # Include BOTH descriptions + required args (helps the model choose correct keys)
    tool_lines = []
    for t in tools_meta:
        name = t["name"]
        desc = t.get("description", "") or ""
        req = schema_map.get(name, {}).get("required", [])
        props = list((schema_map.get(name, {}).get("properties") or {}).keys())
        tool_lines.append(f"- {name}: {desc} | required={req} | keys={props}")
    tool_list_md = "\n".join(tool_lines)

    system = (
        "You are a campus information assistant for university faculty.\n"
        "You MUST choose exactly one tool from the available tools.\n"
        "Return ONLY valid JSON with keys: tool_name, arguments.\n"
        "IMPORTANT: arguments keys MUST match the tool schema keys exactly.\n"
        "No markdown. No extra keys.\n"
    )

    example = '{"tool_name":"find_room","arguments":{"building":"CS","room":"CS-Lab-2"}}'
    user = (
        f"AVAILABLE TOOLS (with schemas):\n{tool_list_md}\n\n"
        f"QUESTION:\n{question}\n\n"
        f"Return JSON like: {example}"
    )

    resp = client.responses.create(
        model=LLM_MODEL,
        input=[{"role": "system", "content": system}, {"role": "user", "content": user}],
        temperature=0.0,
    )

    raw = resp.output_text.strip()
    try:
        obj = json.loads(raw)
        return {**state, "tool_call": obj, "decision_raw": raw, "schema_map": schema_map, "error": None}
    except Exception:
        return {**state, "tool_call": None, "decision_raw": raw, "schema_map": schema_map, "error": "Invalid JSON from decide_tool"}


def normalize_tool_call(state: Dict[str, Any]) -> Dict[str, Any]:
    """Lightweight, deterministic argument fixer for common faculty phrasing.

    This keeps the lesson robust (professors shouldn't fight tool arg formats).
    """
    tool_call = state.get("tool_call") or {}
    tool_name = tool_call.get("tool_name")
    args = tool_call.get("arguments") or {}
    question = state.get("question", "")
    schema_map = state.get("schema_map", {})
    schema = schema_map.get(tool_name, {})
    required = set(schema.get("required", []) or [])

    # --- Heuristics by tool ---
    q_lower = question.lower()

    if tool_name == "find_room":
        # If question contains a token like CS-Lab-2, derive building + room
        room_token = args.get("room") or _extract_room_token(question)
        building = args.get("building")
        if room_token and not building:
            building = room_token.split("-")[0].upper()
        if room_token:
            args = {"building": building or "", "room": room_token}
    elif tool_name == "list_contacts":
        if "it helpdesk" in q_lower or "helpdesk" in q_lower:
            args = {"contact_type": "it_helpdesk"}
        elif "library" in q_lower:
            args = {"contact_type": "library"}
        elif "academic office" in q_lower:
            args = {"contact_type": "academic_office"}
    elif tool_name == "find_timetable":
        cc = args.get("course_code") or _extract_course_code(question)
        wk = args.get("week")
        if wk is None:
            wk = _extract_week(question)
        new_args = {}
        if cc:
            new_args["course_code"] = cc
        if wk is not None:
            new_args["week"] = int(wk)
        args = new_args
    elif tool_name == "get_office_hours":
        sn = args.get("staff_name") or _extract_staff_name(question)
        if sn:
            args = {"staff_name": sn}
    elif tool_name == "find_staff":
        # default: choose query as last name fragment if not provided
        if "query" not in args or not args.get("query"):
            sn = _extract_staff_name(question)
            if sn:
                # use first name token as search query
                args = {"query": sn.split()[-1]}

    # --- Required-fields check (if required missing, keep original but add error hint) ---
    missing = [k for k in required if not args.get(k)]
    if missing:
        return {**state, "tool_call": {"tool_name": tool_name, "arguments": args}, "arg_warning": f"Missing required args: {missing}"}
    return {**state, "tool_call": {"tool_name": tool_name, "arguments": args}}


def _content_to_text(parts) -> str:
    if parts is None:
        return ""
    out = []
    for p in parts:
        out.append(p.text if hasattr(p, "text") else str(p))
    return "\n".join(out)


async def call_tool_node(state: Dict[str, Any]) -> Dict[str, Any]:
    tool_call = state.get("tool_call")
    if not tool_call:
        return {**state, "tool_result": None}

    tool_name = tool_call["tool_name"]
    arguments = tool_call.get("arguments", {}) or {}

    result = await session.call_tool(tool_name, arguments)
    tool_output = (
        result.structuredContent
        if result.structuredContent is not None
        else {"content_text": _content_to_text(result.content)}
    )

    return {
        **state,
        "tool_result": {"tool_name": tool_name, "arguments": arguments, "output": tool_output},
    }


def draft_answer(state: Dict[str, Any]) -> Dict[str, Any]:
    question = state.get("question", "")
    tool_result = state.get("tool_result")
    warning = state.get("arg_warning")

    system = (
        "You are a campus information assistant for university faculty.\n"
        "STRICT RULES:\n"
        "1) Answer ONLY using TOOL_RESULT below.\n"
        "2) If TOOL_RESULT is empty or contains an error, say you cannot find it and suggest what to try next.\n"
        "3) Output exactly in this format:\n"
        "Answer:\n"
        "- ...\n\n"
        "Evidence:\n"
        "- Tool: <tool_name>\n"
        "- Key fields: ...\n\n"
        "Actions / Next steps:\n"
        "- ...\n"
    )

    tool_json = json.dumps(tool_result, indent=2, ensure_ascii=False)
    user = f"QUESTION:\n{question}\n\nTOOL_RESULT:\n{tool_json}"
    if warning:
        user += f"\n\nARG_WARNING:\n{warning}"

    resp = client.responses.create(
        model=LLM_MODEL,
        input=[{"role": "system", "content": system}, {"role": "user", "content": user}],
        temperature=0.0,
    )
    return {**state, "final_answer": resp.output_text}


graph = StateGraph(dict)
graph.add_node("decide_tool", decide_tool)
graph.add_node("normalize_tool_call", normalize_tool_call)
graph.add_node("call_tool", call_tool_node)
graph.add_node("draft_answer", draft_answer)

graph.set_entry_point("decide_tool")
graph.add_edge("decide_tool", "normalize_tool_call")
graph.add_edge("normalize_tool_call", "call_tool")
graph.add_edge("call_tool", "draft_answer")
graph.add_edge("draft_answer", END)

tool_agent = graph.compile()
print("✅ LangGraph tool agent compiled (with argument normalization)")


✅ LangGraph tool agent compiled (with argument normalization)


## 6. Try the tool agent (direct run)

In [7]:
from IPython.display import Markdown, display

async def run_tool_agent(question: str) -> str:
    out = await tool_agent.ainvoke({"question": question, "tools_meta": tools})
    return out["final_answer"]

questions = [
    "Find staff member Amina and give her office and email.",
    "What are the office hours for Dr Amina Rahman?",
    "Where is CS-Lab-2 and what facilities does it have?",
    "Show timetable for SENG3200 week 3.",
    "Give IT helpdesk contact details."
]

for q in questions:
    print("\n" + "="*90)
    print("Q:", q)
    ans = await run_tool_agent(q)
    display(Markdown(ans))


Q: Find staff member Amina and give her office and email.


HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


Answer:
- Dr. Amina Rahman
- Office: CS-3.21
- Email: amina.rahman@uni.example

Evidence:
- Tool: find_staff
- Key fields: name, department, email, office

Actions / Next steps:
- If you need more information about Dr. Amina Rahman or her department, please let me know!


Q: What are the office hours for Dr Amina Rahman?


HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


...

HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


Answer:
- The timetable for SENG3200 in week 3 is as follows:
  - Monday: 10:00-12:00 at LT-1
  - Thursday: 14:00-16:00 at CS-Lab-2

Evidence:
- Tool: find_timetable
- Key fields: course_code: SENG3200, week: 3

Actions / Next steps:
- If you need more information or details about other weeks, please let me know!


Q: Give IT helpdesk contact details.


HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


Answer:
- Name: IT Helpdesk
- Email: helpdesk@uni.example
- Phone: +60-3-5555-1000

Evidence:
- Tool: list_contacts
- Key fields: contact_type, name, email, phone

Actions / Next steps:
- If you need further assistance, please let me know!

## 7. Refactor into a reusable Agent class (`agents_tool.py`)

In [14]:
%%writefile agents_tool.py
from __future__ import annotations

import json
import os
import re
from typing import Any, Dict, List, Optional

from dotenv import load_dotenv
from openai import OpenAI
from langgraph.graph import StateGraph, END

from mcp.shared.memory import create_connected_server_and_client_session
import campus_tools_mcp  # uses the FastMCP app from Lesson 2


def _build_schema_map(tools_meta: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    return {t["name"]: (t.get("inputSchema") or {}) for t in tools_meta}


def _extract_week(text: str) -> Optional[int]:
    m = re.search(r"\bweek\s*(\d+)\b", text, flags=re.I)
    return int(m.group(1)) if m else None


def _extract_course_code(text: str) -> Optional[str]:
    m = re.search(r"\b[A-Z]{3,6}\d{3,5}\b", text)
    return m.group(0) if m else None


def _extract_staff_name(text: str) -> Optional[str]:
    m = re.search(r"\bfor\s+((Dr|Prof|Ms|Mr)\.?\s+[A-Za-z]+(?:\s+[A-Za-z]+){0,3})\b", text)
    return m.group(1).strip() if m else None


def _extract_room_token(text: str) -> Optional[str]:
    m = re.search(r"\bLT-\d+\b|\bCS-Lab-\d+\b|\bLIB-Meeting-\d+\b|\b[A-Za-z]{2,5}-[A-Za-z]+-\d+\b", text)
    return m.group(0) if m else None


class CampusInfoToolAgent:
    """Tool-using campus info agent: LLM selects an MCP tool and answers from tool results only.
    Uses an in-memory MCP session for workshop reliability.
    """

    def __init__(self, model: str = "gpt-4o-mini") -> None:
        load_dotenv()
        if not os.environ.get("OPENAI_API_KEY"):
            raise RuntimeError("OPENAI_API_KEY is not set in the environment.")

        self.client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
        self.model = model

        self.tools_meta: List[Dict[str, Any]] = []
        self.schema_map: Dict[str, Dict[str, Any]] = {}

        self._session = None
        self._session_cm = None

        self._graph = self._build_graph()

    async def initialize(self) -> "CampusInfoToolAgent":
        # In-memory MCP server+client session (no subprocess)
        self._session_cm = create_connected_server_and_client_session(
            campus_tools_mcp.mcp, raise_exceptions=True
        )
        self._session = await self._session_cm.__aenter__()

        tools_obj = await self._session.list_tools()
        self.tools_meta = [
            {"name": t.name, "description": t.description, "inputSchema": t.inputSchema}
            for t in tools_obj.tools
        ]
        self.schema_map = _build_schema_map(self.tools_meta)
        return self

    async def close(self) -> None:
        if self._session_cm is not None:
            await self._session_cm.__aexit__(None, None, None)
            self._session_cm = None
            self._session = None

    # ---------------- LangGraph nodes ----------------
    def _decide_tool(self, state: Dict[str, Any]) -> Dict[str, Any]:
        question = state["question"]

        tool_lines = []
        for t in self.tools_meta:
            name = t["name"]
            desc = t.get("description", "") or ""
            schema = self.schema_map.get(name, {})
            req = schema.get("required", []) or []
            props = list((schema.get("properties") or {}).keys())
            tool_lines.append(f"- {name}: {desc} | required={req} | keys={props}")
        tool_list_md = "\n".join(tool_lines)

        system = (
            "You are a campus information assistant for university faculty.\n"
            "You MUST choose exactly one tool from the available tools.\n"
            "Return ONLY valid JSON with keys: tool_name, arguments.\n"
            "IMPORTANT: argument keys MUST match schema keys exactly.\n"
            "No markdown. No extra keys.\n"
        )

        example = '{"tool_name":"find_room","arguments":{"building":"CS","room":"CS-Lab-2"}}'
        user = (
            f"AVAILABLE TOOLS (with schemas):\n{tool_list_md}\n\n"
            f"QUESTION:\n{question}\n\n"
            f"Return JSON like: {example}"
        )

        resp = self.client.responses.create(
            model=self.model,
            input=[{"role": "system", "content": system}, {"role": "user", "content": user}],
            temperature=0.0,
        )
        raw = resp.output_text.strip()
        try:
            obj = json.loads(raw)
            return {**state, "tool_call": obj, "decision_raw": raw, "error": None}
        except Exception:
            return {**state, "tool_call": None, "decision_raw": raw, "error": "Invalid JSON from decide_tool"}

    def _normalize_tool_call(self, state: Dict[str, Any]) -> Dict[str, Any]:
        tool_call = state.get("tool_call") or {}
        tool_name = tool_call.get("tool_name")
        args = tool_call.get("arguments") or {}
        question = state.get("question", "")
        q_lower = question.lower()

        schema = self.schema_map.get(tool_name, {})
        required = set(schema.get("required", []) or [])

        if tool_name == "find_room":
            room_token = args.get("room") or _extract_room_token(question)
            building = args.get("building")
            if room_token and not building:
                building = room_token.split("-")[0].upper()
            if room_token:
                args = {"building": building or "", "room": room_token}

        elif tool_name == "list_contacts":
            if "it helpdesk" in q_lower or "helpdesk" in q_lower:
                args = {"contact_type": "it_helpdesk"}
            elif "library" in q_lower:
                args = {"contact_type": "library"}
            elif "academic office" in q_lower:
                args = {"contact_type": "academic_office"}

        elif tool_name == "find_timetable":
            cc = args.get("course_code") or _extract_course_code(question)
            wk = args.get("week")
            if wk is None:
                wk = _extract_week(question)
            new_args = {}
            if cc:
                new_args["course_code"] = cc
            if wk is not None:
                new_args["week"] = int(wk)
            args = new_args

        elif tool_name == "get_office_hours":
            sn = args.get("staff_name") or _extract_staff_name(question)
            if sn:
                args = {"staff_name": sn}

        elif tool_name == "find_staff":
            if "query" not in args or not args.get("query"):
                sn = _extract_staff_name(question)
                if sn:
                    args = {"query": sn.split()[-1]}

        missing = [k for k in required if not args.get(k)]
        if missing:
            return {**state, "tool_call": {"tool_name": tool_name, "arguments": args}, "arg_warning": f"Missing required args: {missing}"}
        return {**state, "tool_call": {"tool_name": tool_name, "arguments": args}}

    async def _call_tool(self, state: Dict[str, Any]) -> Dict[str, Any]:
        if self._session is None:
            raise RuntimeError("Agent not initialized. Call await agent.initialize() first.")

        tool_call = state.get("tool_call")
        if not tool_call:
            return {**state, "tool_result": None}

        tool_name = tool_call["tool_name"]
        arguments = tool_call.get("arguments", {}) or {}

        result = await self._session.call_tool(tool_name, arguments)

        def _content_to_text(parts) -> str:
            if parts is None:
                return ""
            out = []
            for p in parts:
                out.append(p.text if hasattr(p, "text") else str(p))
            return "\n".join(out)

        tool_output = (
            result.structuredContent
            if result.structuredContent is not None
            else {"content_text": _content_to_text(result.content)}
        )

        return {**state, "tool_result": {"tool_name": tool_name, "arguments": arguments, "output": tool_output}}

    def _draft_answer(self, state: Dict[str, Any]) -> Dict[str, Any]:
        question = state.get("question", "")
        tool_result = state.get("tool_result")
        warning = state.get("arg_warning")

        system = (
            "You are a campus information assistant for university faculty.\n"
            "STRICT RULES:\n"
            "1) Answer ONLY using TOOL_RESULT below.\n"
            "2) If TOOL_RESULT is empty or contains an error, say you cannot find it and suggest what to try next.\n"
            "3) Output exactly in this format:\n"
            "Answer:\n"
            "- ...\n\n"
            "Evidence:\n"
            "- Tool: <tool_name>\n"
            "- Key fields: ...\n\n"
            "Actions / Next steps:\n"
            "- ...\n"
        )

        tool_json = json.dumps(tool_result, indent=2, ensure_ascii=False)
        user = f"QUESTION:\n{question}\n\nTOOL_RESULT:\n{tool_json}"
        if warning:
            user += f"\n\nARG_WARNING:\n{warning}"

        resp = self.client.responses.create(
            model=self.model,
            input=[{"role": "system", "content": system}, {"role": "user", "content": user}],
            temperature=0.0,
        )
        return {**state, "final_answer": resp.output_text}

    def _build_graph(self):
        g = StateGraph(dict)
        g.add_node("decide_tool", self._decide_tool)
        g.add_node("normalize_tool_call", self._normalize_tool_call)
        g.add_node("call_tool", self._call_tool)
        g.add_node("draft_answer", self._draft_answer)

        g.set_entry_point("decide_tool")
        g.add_edge("decide_tool", "normalize_tool_call")
        g.add_edge("normalize_tool_call", "call_tool")
        g.add_edge("call_tool", "draft_answer")
        g.add_edge("draft_answer", END)
        return g.compile()

    async def answer_query(self, question: str) -> str:
        out = await self._graph.ainvoke({"question": question})
        return out["final_answer"]

Overwriting agents_tool.py


In [15]:
import importlib, agents_tool
importlib.reload(agents_tool)
print("Reloaded from:", agents_tool.__file__)

from agents_tool import CampusInfoToolAgent
from IPython.display import Markdown, display

campus_agent = await CampusInfoToolAgent().initialize()
ans = await campus_agent.answer_query("Give IT helpdesk contact details.")
display(Markdown(ans))

await campus_agent.close()

Reloaded from: /home/robomy/Desktop/THALHATH/5-day-Training/A2A/agents_tool.py


Processing request of type ListToolsRequest
HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
Processing request of type CallToolRequest
HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


Answer:
- IT Helpdesk can be contacted via email at helpdesk@uni.example or by phone at +60-3-5555-1000.

Evidence:
- Tool: list_contacts
- Key fields: contact_type, name, email, phone

Actions / Next steps:
- If you need further assistance or specific IT issues, please reach out to the IT Helpdesk directly.

## 8. Wrap CampusInfoToolAgent as an A2A server (`a2a_campus_info_agent.py`)

In [16]:
%%writefile a2a_campus_info_agent.py
import os
import uvicorn
from dotenv import load_dotenv

from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.apps import A2AStarletteApplication
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore
from a2a.types import AgentCapabilities, AgentCard, AgentSkill
from a2a.utils import new_agent_text_message

from agents_tool import CampusInfoToolAgent


class CampusInfoExecutor(AgentExecutor):
    def __init__(self) -> None:
        self.agent = None

    async def _ensure_init(self):
        if self.agent is None:
            self.agent = await CampusInfoToolAgent().initialize()

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        await self._ensure_init()
        prompt = context.get_user_input()
        response = await self.agent.answer_query(prompt)
        await event_queue.enqueue_event(new_agent_text_message(response))

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        pass


def main() -> None:
    load_dotenv()
    host = os.environ.get("AGENT_HOST", "127.0.0.1")
    port = int(os.environ.get("CAMPUS_INFO_AGENT_PORT", "9998"))

    skill = AgentSkill(
        id="campus_info_tools",
        name="Campus Information Tools",
        description="Finds staff contacts, office hours, timetables, rooms, and helpdesk contacts using MCP tools.",
        tags=["campus", "staff", "timetable", "room", "contacts", "mcp"],
        examples=[
            "Find staff member Amina and give her email and office.",
            "What are the office hours for Dr Amina Rahman?",
            "Show timetable for SENG3200 week 3.",
            "Where is CS-Lab-2 and what facilities does it have?",
            "Give IT helpdesk contact details."
        ],
    )

    agent_card = AgentCard(
        name="CampusInfoToolAgent",
        description="A tool-using campus assistant. It chooses MCP tools and answers only from tool outputs.",
        url=f"http://{host}:{port}/",
        version="1.0.0",
        default_input_modes=["text"],
        default_output_modes=["text"],
        capabilities=AgentCapabilities(streaming=False),
        skills=[skill],
    )

    request_handler = DefaultRequestHandler(
        agent_executor=CampusInfoExecutor(),
        task_store=InMemoryTaskStore(),
    )

    server = A2AStarletteApplication(agent_card=agent_card, http_handler=request_handler)

    print(f"✅ Running CampusInfoToolAgent at http://{host}:{port}/")
    uvicorn.run(server.build(), host=host, port=port)


if __name__ == "__main__":
    main()


Writing a2a_campus_info_agent.py


### 8.1 Run the A2A server (Terminal)

```bash
python a2a_campus_info_agent.py
```

Keep it running for the client test below.

Default URL: `http://127.0.0.1:9998/`


## 9. Test via A2A Client (discovery + send message)

In [19]:
import httpx
from IPython.display import Markdown, display

from a2a.client import ClientConfig, ClientFactory, create_text_message_object
from a2a.types import AgentCard, Artifact, Message, Task
from a2a.utils.message import get_message_text

def display_agent_card(agent_card: AgentCard) -> None:
    def esc(text: str) -> str:
        return str(text).replace("|", r"\|")

    md_parts = [
        "### Agent Card Details",
        "| Property | Value |",
        "| :--- | :--- |",
        f"| **Name** | {esc(agent_card.name)} |",
        f"| **Description** | {esc(agent_card.description)} |",
        f"| **Version** | `{esc(agent_card.version)}` |",
        f"| **URL** | `{esc(agent_card.url)}` |",
        f"| **Protocol Version** | `{esc(agent_card.protocol_version)}` |",
    ]
    if agent_card.skills:
        md_parts.extend(
            [
                "\n#### Skills",
                "| Name | Description | Examples |",
                "| :--- | :--- | :--- |",
            ]
        )
        for skill in agent_card.skills:
            examples_str = (
                "<br>".join(f"• {esc(ex)}" for ex in (skill.examples or []))
                if skill.examples
                else "N/A"
            )
            md_parts.append(
                f"| **{esc(skill.name)}** | {esc(skill.description)} | {examples_str} |"
            )

    display(Markdown("\n".join(md_parts)))

async def call_a2a_agent(prompt: str, host: str, port: int) -> str:
    async with httpx.AsyncClient(timeout=120.0) as httpx_client:
        client = await ClientFactory.connect(
            f"http://{host}:{port}",
            client_config=ClientConfig(httpx_client=httpx_client),
        )
        card = await client.get_card()
        display_agent_card(card)

        message = create_text_message_object(content=prompt)
        responses = client.send_message(message)

        text_content = ""
        async for response in responses:
            if isinstance(response, Message):
                text_content = get_message_text(response)
            elif isinstance(response, tuple):
                task: Task = response[0]
                if task.artifacts:
                    artifact: Artifact = task.artifacts[0]
                    text_content = get_message_text(artifact)
        return text_content

prompt = "Show timetable for SENG3200 week 3."
result = await call_a2a_agent(prompt, host="127.0.0.1", port=9998)
display(Markdown("## Final Response\n---\n" + (result or "*No response received*")))

HTTP Request: GET http://127.0.0.1:9998/.well-known/agent-card.json "HTTP/1.1 200 OK"
Successfully fetched agent card data from http://127.0.0.1:9998/.well-known/agent-card.json: {'capabilities': {'streaming': False}, 'defaultInputModes': ['text'], 'defaultOutputModes': ['text'], 'description': 'A tool-using campus assistant. It chooses MCP tools and answers only from tool outputs.', 'name': 'CampusInfoToolAgent', 'preferredTransport': 'JSONRPC', 'protocolVersion': '0.3.0', 'skills': [{'description': 'Finds staff contacts, office hours, timetables, rooms, and helpdesk contacts using MCP tools.', 'examples': ['Find staff member Amina and give her email and office.', 'What are the office hours for Dr Amina Rahman?', 'Show timetable for SENG3200 week 3.', 'Where is CS-Lab-2 and what facilities does it have?', 'Give IT helpdesk contact details.'], 'id': 'campus_info_tools', 'name': 'Campus Information Tools', 'tags': ['campus', 'staff', 'timetable', 'room', 'contacts', 'mcp']}], 'url': 'ht

### Agent Card Details
| Property | Value |
| :--- | :--- |
| **Name** | CampusInfoToolAgent |
| **Description** | A tool-using campus assistant. It chooses MCP tools and answers only from tool outputs. |
| **Version** | `1.0.0` |
| **URL** | `http://127.0.0.1:9998/` |
| **Protocol Version** | `0.3.0` |

#### Skills
| Name | Description | Examples |
| :--- | :--- | :--- |
| **Campus Information Tools** | Finds staff contacts, office hours, timetables, rooms, and helpdesk contacts using MCP tools. | • Find staff member Amina and give her email and office.<br>• What are the office hours for Dr Amina Rahman?<br>• Show timetable for SENG3200 week 3.<br>• Where is CS-Lab-2 and what facilities does it have?<br>• Give IT helpdesk contact details. |

HTTP Request: POST http://127.0.0.1:9998/ "HTTP/1.1 200 OK"


## Final Response
---
Answer:
- The timetable for SENG3200 in week 3 is as follows:
  - Monday: 10:00-12:00 at LT-1
  - Thursday: 14:00-16:00 at CS-Lab-2

Evidence:
- Tool: find_timetable
- Key fields: course_code, week, day, time, venue

Actions / Next steps:
- If you need more information or details about other weeks, please let me know!

## 10. Mini-orchestrator (two A2A agents for one question)

We assume:
- CoursePolicyAgent server running: `127.0.0.1:9999` (Lesson 1)
- CampusInfoToolAgent server running: `127.0.0.1:9998` (this lesson)

We call both, then merge.


In [20]:
def merge_two_answers(policy_answer: str, campus_answer: str) -> str:
    return (
        "## Merged Faculty Answer\n"
        "---\n"
        "### Policy (Course handbook)\n"
        f"{policy_answer}\n\n"
        "### Campus info (MCP tools)\n"
        f"{campus_answer}\n"
    )

policy_resp = await call_a2a_agent(
    "What is the late submission penalty after 2 days?",
    host="127.0.0.1",
    port=9999,
)

campus_resp = await call_a2a_agent(
    "What are the office hours for Dr Amina Rahman?",
    host="127.0.0.1",
    port=9998,
)

display(Markdown(merge_two_answers(policy_resp, campus_resp)))

HTTP Request: GET http://127.0.0.1:9999/.well-known/agent-card.json "HTTP/1.1 200 OK"
Successfully fetched agent card data from http://127.0.0.1:9999/.well-known/agent-card.json: {'capabilities': {'streaming': False}, 'defaultInputModes': ['text'], 'defaultOutputModes': ['text'], 'description': 'A document-grounded assistant for course policy questions. Answers only using the provided handbook.', 'name': 'CoursePolicyAgent', 'preferredTransport': 'JSONRPC', 'protocolVersion': '0.3.0', 'skills': [{'description': 'Answers questions grounded in the course policy handbook (late penalties, extensions, integrity, AI tool rules, appeals).', 'examples': ['What is the late submission penalty after 2 days?', 'Are AI tools permitted for the Reflection Log?', 'How do students appeal a marking decision?'], 'id': 'course_policy_qna', 'name': 'Course Policy Q&A', 'tags': ['course', 'policy', 'rubric', 'academic integrity']}], 'url': 'http://127.0.0.1:9999/', 'version': '1.0.0'}


### Agent Card Details
| Property | Value |
| :--- | :--- |
| **Name** | CoursePolicyAgent |
| **Description** | A document-grounded assistant for course policy questions. Answers only using the provided handbook. |
| **Version** | `1.0.0` |
| **URL** | `http://127.0.0.1:9999/` |
| **Protocol Version** | `0.3.0` |

#### Skills
| Name | Description | Examples |
| :--- | :--- | :--- |
| **Course Policy Q&A** | Answers questions grounded in the course policy handbook (late penalties, extensions, integrity, AI tool rules, appeals). | • What is the late submission penalty after 2 days?<br>• Are AI tools permitted for the Reflection Log?<br>• How do students appeal a marking decision? |

HTTP Request: POST http://127.0.0.1:9999/ "HTTP/1.1 200 OK"
HTTP Request: GET http://127.0.0.1:9998/.well-known/agent-card.json "HTTP/1.1 200 OK"
Successfully fetched agent card data from http://127.0.0.1:9998/.well-known/agent-card.json: {'capabilities': {'streaming': False}, 'defaultInputModes': ['text'], 'defaultOutputModes': ['text'], 'description': 'A tool-using campus assistant. It chooses MCP tools and answers only from tool outputs.', 'name': 'CampusInfoToolAgent', 'preferredTransport': 'JSONRPC', 'protocolVersion': '0.3.0', 'skills': [{'description': 'Finds staff contacts, office hours, timetables, rooms, and helpdesk contacts using MCP tools.', 'examples': ['Find staff member Amina and give her email and office.', 'What are the office hours for Dr Amina Rahman?', 'Show timetable for SENG3200 week 3.', 'Where is CS-Lab-2 and what facilities does it have?', 'Give IT helpdesk contact details.'], 'id': 'campus_info_tools', 'name': 'Campus Information Tools', 'tags': ['campus', 's

### Agent Card Details
| Property | Value |
| :--- | :--- |
| **Name** | CampusInfoToolAgent |
| **Description** | A tool-using campus assistant. It chooses MCP tools and answers only from tool outputs. |
| **Version** | `1.0.0` |
| **URL** | `http://127.0.0.1:9998/` |
| **Protocol Version** | `0.3.0` |

#### Skills
| Name | Description | Examples |
| :--- | :--- | :--- |
| **Campus Information Tools** | Finds staff contacts, office hours, timetables, rooms, and helpdesk contacts using MCP tools. | • Find staff member Amina and give her email and office.<br>• What are the office hours for Dr Amina Rahman?<br>• Show timetable for SENG3200 week 3.<br>• Where is CS-Lab-2 and what facilities does it have?<br>• Give IT helpdesk contact details. |

HTTP Request: POST http://127.0.0.1:9998/ "HTTP/1.1 200 OK"


## Merged Faculty Answer
---
### Policy (Course handbook)
Answer: The late submission penalty after 2 days is a 20% deduction, resulting in a maximum of 80% of the total marks available.

Evidence:
- "2 days (25–48 hrs) 20% deducted 80% of total marks" (Page 4)
- "Late penalties are applied automatically by the LMS at the time of marking." (Page 4)

### Campus info (MCP tools)
Answer:
- Dr. Amina Rahman's office hours are as follows:
  - Tuesday: 14:00-16:00 (In-person at CS-3.21)
  - Thursday: 10:00-11:00 (Online via MS Teams)

Evidence:
- Tool: get_office_hours
- Key fields: staff_name, office_hours

Actions / Next steps:
- If you need further assistance or information, please let me know!


## 11. Exercises (for faculty participants)

1) Improve tool selection:
- In `decide_tool`, add: “If question asks for contact details, prefer list_contacts.”

2) Add a guardrail:
- If the model chooses a tool name not in the tool list, re-ask once.

3) Extend dataset:
- Add one more staff member + their office hours and test.

> Lesson 5 upgrades this into a full router + fallback.
